<h1> Lista 2 - CIN7915-07342 (20251) - Data Science </h1>

João Pedro Becker Schneider\
<b>objetivos:</b>
1. Elabore um workflow para um sistema de recomendação.
2. Use o dataset Jazz.csv disponibilizado no Moodle.

In [ ]:
#importaçao das bibliotecas
import pandas as pd
import nltk
import re
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

#pacotes do NLTK
#nltk.download('punkt')
nltk.download('punkt_tab') #tokenizador atual do nltk
nltk.download('stopwords') #dicionarios de stopwords
nltk.download('averaged_perceptron_tagger_eng') #modelo de POS tagging (Part-of-Speech).

In [ ]:
#importando dataset
data = pd.read_csv("jazz.csv")

#data.head()
#data.sample()

In [ ]:
#faz o pre-processamento da coluna 'bio'
def preprocess(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()  #converte texto para minusculo
    text = re.sub(r'[^\w\s]', '', text)  #apaga pontuacoes
    tokens = nltk.word_tokenize(text) #tokeniza o que sobra

    stopwords = set(nltk.corpus.stopwords.words('english')) #atribui stopwords
    tokens = [t for t in tokens if t not in stopwords and len(t) > 2]  # remove stopwords e palavras pequenas

    tagged = nltk.pos_tag(tokens)
    tokens = [word for word, pos in tagged if pos.startswith('NN')]  # filtra substantivos (tag filtering)

    stemmer = nltk.SnowballStemmer('english')
    tokens = [stemmer.stem(t) for t in tokens]  #stemming

    return ' '.join(tokens)

#cria coluna nova com o que foi processado de bio
data['processed'] = data['bio'].apply(preprocess)

data[['name', 'processed']].head()

In [ ]:
#intancia vetorizador TF - IDF
vectorizer = TfidfVectorizer()

# Cada linha vira um vetor numérico, com pesos TF-IDF para cada termo
tfidf_matrix = vectorizer.fit_transform(data['processed'])

# (número de documentos, número de termos únicos)
print(f"Shape da matriz TF-IDF: {tfidf_matrix.shape}")

In [ ]:
# Calcula a matriz de similaridade entre os artistas com base nos vetores TF-IDF
similarity_matrix = cosine_similarity(tfidf_matrix)

def recommend(index, top_n=5): #retorna os artistas mais similares ao artista no índice fornecido onde top_n é a quantidade de recomendações
    similarity_scores = list(enumerate(similarity_matrix[index]))
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
    recommended_indices = [i for i, score in similarity_scores[1:top_n+1]]
    return data.iloc[recommended_indices][['name', 'bio']]

#teste
# print("Recomendações para:", data.loc[0, 'name'])
# print(recommend(0))

##Exemplo de Aplicação do modelo de Recomendação

In [ ]:
#imprime os nomes dos artistas
print("Artistas disponíveis:")
for name in data['name']:
    print(f" - {name}")

#pega indice correspondente
def get_index_from_name(name):
    matches = data[data['name'].str.contains(name, case=False, na=False)]
    if matches.empty:
        return None
    return matches.index[0]

#captura escolha do usuário para usar no sistema
user_input = input("Digite o nome de um artista de jazz para ver recomendações: ")
index = get_index_from_name(user_input)

#usa o modelo de recomendaçao
if index is not None:
    print(f"Artistas parecidos com {data.loc[index, 'name']}:\n")
    display(recommend(index, top_n=3))
else:
    print("Artista não encontrado.")